# PlanForge sheet-view detector (head 0)

Finds each **plan view**, the **title block** and **schedule tables** on a whole drawing sheet, so the floor-plan
model can read each floor of a G+1 / G+2 sheet separately.

Training data is composed here from real plan drawings (ResPlan renders, plus Structured3D if attached): 1-4 plans
per sheet with titles (GROUND FLOOR PLAN ...), border, title block, notes and tables, so every box is exact.

**Before running:** Settings -> Accelerator **GPU T4 x2**, Internet **On**; Kaggle Secret **HF_TOKEN** (write).
Optional: attach your **private** Structured3D dataset (Structured3D_annotation_3d.zip) for more plan styles.
Output: private HF model repo `<you>/planforge-sheet-views` with `checkpoints/best.pt`, `sheet_views.onnx`,
`sheet_views_meta.json`.

In [ ]:
# ===================== CONFIG =====================
import os, sys, time, json, shutil, subprocess, random, glob
NOTEBOOK_START = time.time()
MODEL_REPO_NAME = "planforge-sheet-views"     # private HF model repo
BASE_MODEL      = "yolo11s.pt"
N_SHEETS        = 6000                         # composed sheets (10% validation)
N_RESPLAN       = 5000                         # ResPlan plans rendered as the views
N_S3D           = 2500                         # Structured3D houses (only if attached; private, never uploaded)
IMGSZ, BATCH    = 1280, 16
MAX_EPOCHS      = 150
TIME_BUDGET_HOURS = 6.0
SHEET_MAX_SIDE  = 1600                          # sheets are saved at this size (the detector sees the whole sheet)
SRC_DIR, DS_DIR, PLAN_DIR, RUNS_DIR = "/kaggle/working/sheet_src", "/tmp/sheet_ds", "/tmp/sheet_plans", "/tmp/sheet_runs"
for d in (SRC_DIR, DS_DIR, PLAN_DIR, RUNS_DIR): os.makedirs(d, exist_ok=True)
print("config ok")

In [ ]:
!pip install -q "ultralytics==8.3.253" "huggingface_hub>=0.25" shapely onnx onnxslim 2>shapely onnx 2>&11 | grep -v "already satisfied" | tail -2

In [ ]:
%%writefile /kaggle/working/sheet_src/fp_common.py
"""Shared code: classes, cached dataset + augmentation, model, losses, metrics."""
import json
import math
import os
import random

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(0)

# ----------------------------------------------------------------------------- classes
# Room head (from the official CubiCasa5K "rooms_selected" mapping)
ROOM_CLASSES = ["Background", "Outdoor", "Wall", "Kitchen", "LivingRoom", "Bedroom",
                "Bath", "Entry/Hallway", "Railing", "Storage", "Garage", "OtherRoom"]
# Icon head (official "icons_selected" mapping) + Stairs (parsed separately, id 11)
ICON_CLASSES = ["NoIcon", "Window", "Door", "Closet", "ElectricalAppliance", "Toilet",
                "Sink", "SaunaBench", "Fireplace", "Bathtub", "Chimney", "Stairs"]
# Junction heatmaps (orientation-free so flips/rotations stay valid)
HEAT_CLASSES = ["WallJunction", "OpeningEndpoint", "IconCorner"]

# v2 heads (appended after the heatmaps, so channels 0-26 keep their meaning and older engines still read them)
#   wall type: on wall pixels only. Learned from every source that labels the outside (rooms + background):
#              a wall the outside touches is exterior. Unknown (IGNORE) elsewhere and for stroke-only sources.
#   level:     on floor pixels only. CubiCasa5K marks raised floor ("Elevated") and voids over a lower floor
#              ("OpenToBelow"); Structured3D floors are all normal. Unknown for the other sources.
WALLTYPE_CLASSES = ["Interior", "Exterior"]
LEVEL_CLASSES = ["Normal", "Raised", "OpenToBelow"]

N_ROOM, N_ICON, N_HEAT = len(ROOM_CLASSES), len(ICON_CLASSES), len(HEAT_CLASSES)
N_WT, N_LV = len(WALLTYPE_CLASSES), len(LEVEL_CLASSES)
N_BASE = N_ROOM + N_ICON + N_HEAT           # 27: the v1 model
N_OUT = N_BASE + N_WT + N_LV                # 32
# sources whose room labels include the outside (background / outdoor), so wall types can be derived
WALLTYPE_SOURCES = {"cubicasa", "resplan", "synthetic", "structured3d"}
IGNORE = 255
NOT_WALL = 254       # room target: "known not wall, room type unknown" (sources that only label walls)
# Rotation augmentation: plans with angled wings mostly use these angles; any angle is also trained
ROT_ANGLES = (30.0, 45.0, 60.0)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255.0
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255.0


# ----------------------------------------------------------------------------- heatmaps
def render_heatmaps(points, h, w, sigma):
    """points: (N,3) float array [x, y, type]. Returns (N_HEAT, h, w) float32, peak exactly 1."""
    heat = np.zeros((N_HEAT, h, w), dtype=np.float32)
    if points is None or len(points) == 0:
        return heat
    r = int(math.ceil(3 * sigma))
    ax = np.arange(-r, r + 1, dtype=np.float32)
    g = np.exp(-(ax[None, :] ** 2 + ax[:, None] ** 2) / (2 * sigma * sigma)).astype(np.float32)
    for x, y, t in points:
        cx, cy, t = int(round(float(x))), int(round(float(y))), int(t)
        if not (0 <= cx < w and 0 <= cy < h) or not (0 <= t < N_HEAT):
            continue
        x0, x1 = max(0, cx - r), min(w, cx + r + 1)
        y0, y1 = max(0, cy - r), min(h, cy + r + 1)
        patch = g[y0 - (cy - r):y1 - (cy - r), x0 - (cx - r):x1 - (cx - r)]
        np.maximum(heat[t, y0:y1, x0:x1], patch, out=heat[t, y0:y1, x0:x1])
    return heat


# ----------------------------------------------------------------------------- dataset
class CachedFloorplans(torch.utils.data.Dataset):
    """Reads the preprocessed cache: <root>/data/<id>_img.png, _lbl.png (B=room, G=icon), _pts.npy"""

    def __init__(self, root, split, train, crop=768, sigma=3.0, scale_range=(0.6, 1.8), rot_prob=0.6,
                 fixed_angle=None, sources=None, limit=None):
        """rot_prob: share of training crops rotated by a non-90-degree angle (30/45/60 +-3 deg, or any angle).
        fixed_angle: evaluation only - every sample rotated by exactly this angle (rotated-plan validation)."""
        with open(os.path.join(root, "index.json")) as f:
            index = json.load(f)
        self.items = index[split]
        if sources is not None:                     # e.g. ["cubicasa"]; entries without "source" are CubiCasa5K
            self.items = [it for it in self.items if it.get("source", "cubicasa") in sources]
        if limit:
            self.items = self.items[:limit]
        self.root = os.path.join(root, "data")
        self.train, self.crop, self.sigma, self.scale_range = train, crop, sigma, scale_range
        self.rot_prob, self.fixed_angle = rot_prob, fixed_angle

    def __len__(self):
        return len(self.items)

    def source_of(self, i):
        return self.items[i].get("source", "cubicasa")

    def _load(self, i):
        it = self.items[i]
        sid = it["id"]
        img = cv2.imread(os.path.join(self.root, sid + "_img.png"), cv2.IMREAD_COLOR)
        lbl = cv2.imread(os.path.join(self.root, sid + "_lbl.png"), cv2.IMREAD_UNCHANGED)
        if img is None or lbl is None:
            raise RuntimeError(f"Corrupt cache entry {sid}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pp = os.path.join(self.root, sid + "_pts.npy")
        # sources without junction labels have no _pts file: their heat target is "unknown", not "none"
        pts = np.load(pp).astype(np.float32).reshape(-1, 3) if os.path.exists(pp) else None
        room, icon = lbl[:, :, 0].copy(), lbl[:, :, 1].copy()
        # third label channel = floor level classes, only for entries written with it (index flag "level")
        level = lbl[:, :, 2].copy() if it.get("level") else np.full(room.shape, IGNORE, np.uint8)
        wtype = wall_type_labels(room) if it.get("source", "cubicasa") in WALLTYPE_SOURCES \
            else np.full(room.shape, IGNORE, np.uint8)
        return img, [room, icon, wtype, level], pts

    def __getitem__(self, i):
        img, maps, pts = self._load(i)
        has_heat = pts is not None
        if pts is None:
            pts = np.zeros((0, 3), np.float32)
        if self.train:
            img, maps, pts = self._augment(img, maps, pts)
        elif self.fixed_angle is not None:
            img, maps, pts = self._rotate_whole(img, maps, pts, self.fixed_angle)
            img, maps = self._pad32(img, maps)
        else:
            img, maps = self._pad32(img, maps)
        h, w = maps[0].shape
        heat = render_heatmaps(pts, h, w, self.sigma) if has_heat else np.full((N_HEAT, h, w), -1.0, np.float32)
        x = (img.astype(np.float32) - MEAN) / STD
        t = [torch.from_numpy(np.ascontiguousarray(m)).long() for m in maps]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))), t[0], t[1],
                torch.from_numpy(heat), t[2], t[3])

    @staticmethod
    def _pad32(img, maps):
        h, w = maps[0].shape
        ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
        if ph or pw:
            img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=(255, 255, 255))
            maps = [cv2.copyMakeBorder(m, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE) for m in maps]
        return img, maps

    @staticmethod
    def _rotate_whole(img, maps, pts, angle):
        """Rotate a whole plan by `angle` degrees on a canvas that fits it (evaluation on rotated plans)."""
        h, w = maps[0].shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        c, s_ = abs(M[0, 0]), abs(M[0, 1])
        W2, H2 = int(math.ceil(h * s_ + w * c)), int(math.ceil(h * c + w * s_))
        M[0, 2] += W2 / 2 - w / 2
        M[1, 2] += H2 / 2 - h / 2
        img = cv2.warpAffine(img, M, (W2, H2), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
        maps = [cv2.warpAffine(m, M, (W2, H2), flags=cv2.INTER_NEAREST, borderValue=IGNORE) for m in maps]
        if len(pts):
            pts = pts.copy()
            xy = pts[:, :2] @ M[:, :2].T + M[:, 2]
            pts[:, 0], pts[:, 1] = xy[:, 0], xy[:, 1]
        return img, maps, pts

    def _augment(self, img, maps, pts):
        S = self.crop
        h, w = maps[0].shape
        lo, hi = self.scale_range
        s = math.exp(random.uniform(math.log(lo), math.log(hi)))
        src = S / s  # crop size in source pixels
        x0 = random.uniform(0, w - src) if w > src else -random.uniform(0, src - w)
        y0 = random.uniform(0, h - src) if h > src else -random.uniform(0, src - h)
        M = np.array([[s, 0, -x0 * s], [0, s, -y0 * s]], dtype=np.float32)
        if random.random() < self.rot_prob:
            # angled walls: rotate the crop about its centre (the 90-degree turns / flips below add the rest,
            # so 30 also covers 120, 210, 300 and with the flip -30 ...). Corners outside the plan stay
            # white paper and are ignored in the loss.
            a = random.choice(ROT_ANGLES) + random.uniform(-3, 3) if random.random() < 0.7 else random.uniform(0, 90)
            R = cv2.getRotationMatrix2D((S / 2, S / 2), a, 1.0).astype(np.float32)
            M = (np.vstack([R, [0, 0, 1]]) @ np.vstack([M, [0, 0, 1]]))[:2].astype(np.float32)
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
        img = cv2.warpAffine(img, M, (S, S), flags=interp, borderMode=cv2.BORDER_CONSTANT,
                             borderValue=(255, 255, 255))
        maps = [cv2.warpAffine(m, M, (S, S), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT,
                               borderValue=IGNORE) for m in maps]
        if len(pts):
            pts = pts.copy()
            xy = pts[:, :2] @ M[:, :2].T + M[:, 2]
            pts[:, 0], pts[:, 1] = xy[:, 0], xy[:, 1]
        # rotations / flips (all heads are orientation-free, so labels stay valid)
        k = random.randint(0, 3)
        for _ in range(k):  # 90 deg counter-clockwise
            img, maps = np.rot90(img), [np.rot90(m) for m in maps]
            if len(pts):
                pts[:, 0], pts[:, 1] = pts[:, 1].copy(), (S - 1) - pts[:, 0]
        if random.random() < 0.5:
            img, maps = img[:, ::-1], [m[:, ::-1] for m in maps]
            if len(pts):
                pts[:, 0] = (S - 1) - pts[:, 0]
        img = np.ascontiguousarray(img).astype(np.float32)
        # photometric
        if random.random() < 0.3:
            g = img.mean(axis=2, keepdims=True)
            img = np.repeat(g, 3, axis=2)
        if random.random() < 0.8:
            img = (img - 128.0) * random.uniform(0.75, 1.25) + 128.0 + random.uniform(-25, 25)
        if random.random() < 0.2:
            img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.3, 1.0))
        if random.random() < 0.25:
            img = img + np.random.normal(0, random.uniform(2, 8), img.shape).astype(np.float32)
        img = np.clip(img, 0, 255).astype(np.uint8)
        return img, [np.ascontiguousarray(m) for m in maps], pts


def wall_type_labels(room):
    """Wall pixels -> 0 interior / 1 exterior; IGNORE off walls. A wall is exterior where the outside (background
    connected to the page edge, or an outdoor area) lies within about one wall thickness of it."""
    WALL, BG, OUT = ROOM_CLASSES.index("Wall"), ROOM_CLASSES.index("Background"), ROOM_CLASSES.index("Outdoor")
    out = np.full(room.shape, IGNORE, np.uint8)
    wall = room == WALL
    if not wall.any():
        return out
    bg = (room == BG).astype(np.uint8)
    n, cc = cv2.connectedComponents(bg, connectivity=4)
    edge = np.unique(np.concatenate([cc[0], cc[-1], cc[:, 0], cc[:, -1]]))
    outside = np.isin(cc, edge[edge > 0]) | (room == OUT)
    if not outside.any():
        out[wall] = IGNORE                       # a crop with no outside in it: no way to tell
        return out
    # local wall thickness at each pixel (twice the largest inscribed radius nearby) vs distance to the outside
    dt = cv2.distanceTransform(wall.astype(np.uint8), cv2.DIST_L2, 3)
    k = max(3, int(2 * np.percentile(dt[wall], 95)) * 2 + 1)
    thick = 2.0 * cv2.dilate(dt, np.ones((k, k), np.uint8))
    d_out = cv2.distanceTransform((~outside).astype(np.uint8), cv2.DIST_L2, 3)
    near = d_out <= thick + 1.5
    out[wall] = 0
    out[wall & near] = 1
    return out


# ----------------------------------------------------------------------------- model
def build_model(encoder="tu-efficientnet_b3", weights="imagenet"):
    import segmentation_models_pytorch as smp
    model = smp.Unet(encoder_name=encoder, encoder_weights=weights, classes=N_OUT,
                     decoder_attention_type="scse")
    # focal-loss prior for heatmap logits (CenterNet init)
    head = [m for m in model.segmentation_head.modules() if isinstance(m, nn.Conv2d)][-1]
    with torch.no_grad():
        head.bias[N_ROOM + N_ICON:N_BASE].fill_(-2.19)
    # Some smp decoder blocks own parameters that never get gradients (e.g. the attention on
    # the missing skip of the last block). DDP aborts on those, so detect + freeze them.
    saved = {k: v.clone() for k, v in model.state_dict().items()}  # keep BN stats untouched
    model.train()
    model(torch.randn(2, 3, 64, 64)).float().sum().backward()
    for p in model.parameters():
        if p.grad is None:
            p.requires_grad_(False)
        p.grad = None
    model.load_state_dict(saved)
    return model


def split_logits(out):
    return out[:, :N_ROOM], out[:, N_ROOM:N_ROOM + N_ICON], out[:, N_ROOM + N_ICON:N_BASE]


def split_aux(out):
    """v2 heads: (wall type logits, level logits)."""
    return out[:, N_BASE:N_BASE + N_WT], out[:, N_BASE + N_WT:N_BASE + N_WT + N_LV]


def expand_head(state, model):
    """Weights of a model with fewer output channels (the 27-channel v1 model) -> this model's state dict:
    every layer copied, the segmentation head's first channels copied and the new channels left as initialised.
    Returns (state dict, number of channels copied) or (state, None) when nothing needed changing."""
    own = model.state_dict()
    out, grown = dict(state), None
    for k, v in state.items():
        if k in own and own[k].shape != v.shape and k.startswith("segmentation_head") and \
                own[k].shape[1:] == v.shape[1:] and own[k].shape[0] > v.shape[0]:
            w = own[k].clone()
            w[:v.shape[0]] = v
            out[k], grown = w, int(v.shape[0])
    return out, grown


# ----------------------------------------------------------------------------- losses
def dice_loss(logits, target, n_cls, ignore=IGNORE, eps=1.0):
    valid = (target != ignore)
    t = torch.where(valid, target, torch.zeros_like(target))
    prob = logits.softmax(1) * valid.unsqueeze(1)
    oh = F.one_hot(t, n_cls).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
    dims = (0, 2, 3)
    inter = (prob * oh).sum(dims)
    card = prob.sum(dims) + oh.sum(dims)
    present = oh.sum(dims) > 0  # only classes present in the batch
    dice = (2 * inter + eps) / (card + eps)
    if present.any():
        return 1 - dice[present].mean()
    return logits.sum() * 0.0


def focal_ce(logits, target, gamma=2.0, ignore=IGNORE):
    ce = F.cross_entropy(logits, target, ignore_index=ignore, reduction="none")
    valid = (target != ignore)
    if not valid.any():
        return logits.sum() * 0.0
    pt = torch.exp(-ce)
    return (((1 - pt) ** gamma) * ce)[valid].mean()


def heat_focal(logits, gt, alpha=2.0, beta=4.0):
    keep = gt[:, 0, 0, 0] >= 0                     # samples whose source labels junctions (else target is -1)
    if not keep.any():
        return logits.sum() * 0.0
    logits, gt = logits[keep], gt[keep]
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = gt.ge(0.999).float()
    neg = 1 - pos
    pos_loss = torch.log(pred) * (1 - pred) ** alpha * pos
    neg_loss = torch.log(1 - pred) * pred ** alpha * (1 - gt) ** beta * neg
    n_pos = pos.sum().clamp(min=1.0)
    return -(pos_loss.sum() + neg_loss.sum()) / n_pos


class MultiTaskLoss(nn.Module):
    def __init__(self, w_room=1.0, w_icon=1.0, w_heat=1.0, w_wtype=0.5, w_level=0.5):
        super().__init__()
        self.w = (w_room, w_icon, w_heat, w_wtype, w_level)

    def forward(self, out, room, icon, heat, wtype=None, level=None):
        r, i, h = [t.float() for t in split_logits(out)]
        not_wall = room == NOT_WALL
        room = torch.where(not_wall, torch.full_like(room, IGNORE), room)
        l_room = F.cross_entropy(r, room, ignore_index=IGNORE) + dice_loss(r, room, N_ROOM)
        if not_wall.any():                          # "not a wall, room type unknown": push only P(wall) down
            p_wall = torch.softmax(r, 1)[:, ROOM_CLASSES.index("Wall")]
            l_room = l_room + (-torch.log((1 - p_wall[not_wall]).clamp(min=1e-6))).mean()
        l_icon = focal_ce(i, icon) + dice_loss(i, icon, N_ICON)
        l_heat = heat_focal(h, heat)
        total = self.w[0] * l_room + self.w[1] * l_icon + self.w[2] * l_heat
        parts = {"room": l_room.detach(), "icon": l_icon.detach(), "heat": l_heat.detach()}
        if out.shape[1] >= N_OUT and wtype is not None:
            wt, lv = [t.float() for t in split_aux(out)]
            l_wt = focal_ce(wt, wtype)              # 0 when the batch has no labelled wall (IGNORE everywhere)
            l_lv = focal_ce(lv, level) + (dice_loss(lv, level, N_LV) if (level != IGNORE).any() else lv.sum() * 0)
            total = total + self.w[3] * l_wt + self.w[4] * l_lv
            parts.update(wtype=l_wt.detach(), level=l_lv.detach())
        return total, parts


# ----------------------------------------------------------------------------- metrics
def update_confusion(cm, pred, target, n_cls, ignore=IGNORE):
    valid = (target != ignore) & (target != NOT_WALL)
    idx = target[valid] * n_cls + pred[valid]
    cm += torch.bincount(idx, minlength=n_cls * n_cls).reshape(n_cls, n_cls)
    return cm


def iou_from_confusion(cm):
    cm = cm.double()
    tp = cm.diag()
    denom = cm.sum(0) + cm.sum(1) - tp
    iou = torch.where(denom > 0, tp / denom.clamp(min=1), torch.full_like(tp, float("nan")))
    return iou  # nan for classes absent from both pred and gt


class EMA:
    """Exponential moving average of weights (fp32), with warmup-aware decay."""

    def __init__(self, model, decay=0.999):
        import copy
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


class SourceWeightedSampler(torch.utils.data.Sampler):
    """Each epoch draws `num_samples` items with per-source shares (e.g. CubiCasa 0.4, ResPlan 0.25, synthetic
    angled 0.2, FloorPlanCAD 0.15), so a big source does not drown the others; sharded across DDP ranks."""

    def __init__(self, ds, weights, num_samples, rank=0, world=1, seed=0):
        self.groups = {}
        for i in range(len(ds)):
            self.groups.setdefault(ds.source_of(i), []).append(i)
        w = {k: float(weights.get(k, 0.0)) for k in self.groups}
        tot = sum(w.values()) or 1.0
        self.share = {k: v / tot for k, v in w.items() if v > 0}
        self.num_samples = num_samples - num_samples % world
        self.rank, self.world, self.seed, self.epoch = rank, world, seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        g = random.Random(self.seed + self.epoch)
        picks = []
        for k, sh in self.share.items():
            n = int(round(sh * self.num_samples))
            pool = self.groups[k]
            picks += [pool[g.randrange(len(pool))] for _ in range(n)] if n > len(pool) else g.sample(pool, n)
        g.shuffle(picks)
        picks = picks[:self.num_samples]
        picks += picks[:self.num_samples - len(picks)]
        return iter(picks[self.rank::self.world])

    def __len__(self):
        return self.num_samples // self.world


In [ ]:
%%writefile /kaggle/working/sheet_src/fp_render.py
"""Extra training data for the floor-plan model, written into the same cache as CubiCasa5K.

Sources
  resplan     ResPlan (17k South-Asian residential plans, CC BY 4.0): vector geometry, rendered here as drawings
  synthetic   angled buildings composed from ResPlan wings at 30 / 45 / 60 degrees around a hub (with curved
              fronts), rendered the same way - exact labels for the plans no dataset has enough of
  floorplancad  see fp_extra_fpcad.py (walls, doors, windows, stairs, sanitary from real CAD drawings)

Every plan is drawn in a randomly picked style (solid / hatched / outlined walls, line weights, door swings,
window marks, stair treads, room names and sizes the way Indian plans write them, dimension strings, paper tone,
scan blur) so the model learns the plan, not one drawing style.

Labels follow the cache format: <id>_lbl.png  B = room class, G = icon class (fp_common ids). What a source does
not know is IGNORE (255), so it teaches nothing there: untyped interior space, the inside of a door opening (room
head), the swing area of a door (icon head). These sources have no junction labels (no _pts.npy).
"""
import json
import math
import os
import pickle
import random

import cv2
import numpy as np

from fp_common import ROOM_CLASSES, ICON_CLASSES, IGNORE

R = {n: i for i, n in enumerate(ROOM_CLASSES)}
I = {n: i for i, n in enumerate(ICON_CLASSES)}

# ResPlan layer -> room class (None: icon only / not a room)
RESPLAN_ROOMS = {"living": "LivingRoom", "bedroom": "Bedroom", "kitchen": "Kitchen", "bathroom": "Bath",
                 "balcony": "Outdoor", "garden": "Outdoor", "pool": "Outdoor", "parking": "Outdoor",
                 "storage": "Storage"}
ROOM_NAMES = {
    "LivingRoom": ["LIVING", "HALL", "LIVING ROOM", "DRAWING", "FAMILY LOUNGE", "LOUNGE"],
    "Bedroom": ["BEDROOM", "BED ROOM", "M. BEDROOM", "BED RM", "GUEST ROOM", "KIDS ROOM"],
    "Kitchen": ["KITCHEN", "KIT.", "KITCHEN & DINING", "PANTRY"],
    "Bath": ["TOILET", "BATH", "W.C.", "T & B", "WASH", "ATT. TOILET"],
    "Outdoor": ["BALCONY", "SIT OUT", "VERANDAH", "GARDEN", "OPEN TERRACE", "PORTICO"],
    "Storage": ["STORE", "STORE ROOM", "UTILITY"],
    "Entry/Hallway": ["LOBBY", "FOYER", "PASSAGE", "CORRIDOR", "RECEPTION"],
}
SAFE_PICKLE = {("shapely.io", "from_wkb"), ("numpy._core.multiarray", "scalar"), ("numpy", "dtype"),
               ("numpy.core.multiarray", "scalar")}


class _SafeUnpickler(pickle.Unpickler):
    """ResPlan ships as a pickle: allow only the shapely / numpy constructors it needs (no code execution)."""

    def find_class(self, module, name):
        if (module, name) not in SAFE_PICKLE:
            raise pickle.UnpicklingError("blocked in dataset pickle: {}.{}".format(module, name))
        return super().find_class(module, name)


def load_resplan(pkl_path):
    with open(pkl_path, "rb") as f:
        return _SafeUnpickler(f).load()


# ----------------------------------------------------------------------------- geometry
def _polys(g):
    if g is None or getattr(g, "is_empty", True):
        return []
    if g.geom_type == "Polygon":
        return [g]
    if hasattr(g, "geoms"):
        return [p for q in g.geoms for p in _polys(q)]
    return []


def resplan_plan(p):
    """ResPlan record -> plan dict used by the renderer (units: ResPlan units, metres per unit in 'mpu')."""
    from shapely.ops import unary_union
    rooms = []
    for key, cls in RESPLAN_ROOMS.items():
        rooms += [(cls, q) for q in _polys(p.get(key))]
    stairs = _polys(p.get("stair"))
    doors = _polys(p.get("door")) + _polys(p.get("front_door"))
    walls = unary_union(_polys(p.get("wall")))
    inner = unary_union(_polys(p.get("inner")))
    gross = unary_union([inner, walls]).area
    mpu = math.sqrt(float(p.get("area", 0)) / gross) if gross > 0 and p.get("area") else 0.05
    return {"walls": walls, "rooms": rooms, "doors": doors, "windows": _polys(p.get("window")),
            "stairs": stairs, "inner": inner, "mpu": mpu, "id": "rp{}".format(p.get("id"))}


def _transform(plan, fn):
    out = dict(plan)
    out["walls"] = fn(plan["walls"])
    out["inner"] = fn(plan["inner"])
    out["rooms"] = [(c, fn(q)) for c, q in plan["rooms"]]
    for k in ("doors", "windows", "stairs"):
        out[k] = [fn(q) for q in plan[k]]
    return out


def synthetic_angled(parts, rng):
    """2-3 ResPlan plans as wings, each turned by 30 / 45 / 60 degrees around a shared hub (lobby), sometimes
    with a curved front - the kind of building (school, office, clubhouse) the axis-aligned datasets lack."""
    from shapely import affinity
    from shapely.geometry import Point, Polygon
    from shapely.ops import unary_union
    mpu = parts[0]["mpu"]
    wd = max(parts[0]["walls"].area / max(parts[0]["walls"].length / 2, 1e-6), 2.0)   # wall depth (units)
    n = len(parts)
    step = rng.choice([30, 45, 60])
    base = rng.uniform(0, 360)
    angles = [base + (k - (n - 1) / 2) * 2 * step for k in range(n)]
    wings = []
    radius = 0.0
    for pl in parts:
        b = unary_union([pl["walls"], pl["inner"]])
        c = b.centroid
        pl = _transform(pl, lambda g, c=c: affinity.translate(g, -c.x, -c.y))
        radius = max(radius, math.sqrt(b.area) * 0.75)
        wings.append(pl)
    placed = []
    for pl, a in zip(wings, angles):
        r = radius
        for _ in range(40):          # push the wing out along its direction until it clears the others
            dx, dy = r * math.cos(math.radians(a)), r * math.sin(math.radians(a))
            cand = _transform(pl, lambda g, a=a, dx=dx, dy=dy: affinity.translate(
                affinity.rotate(g, a - 90, origin=(0, 0)), dx, dy))
            foot = unary_union([cand["walls"], cand["inner"]]).buffer(wd)
            if all(not foot.intersects(unary_union([q["walls"], q["inner"]])) for q in placed) and \
                    not foot.contains(Point(0, 0)):
                break
            r *= 1.12
        placed.append(cand)
    feet = unary_union([unary_union([q["walls"], q["inner"]]) for q in placed])
    hull = feet.convex_hull
    if rng.random() < 0.4:           # curved front: a circular bulge on the hub
        hull = unary_union([hull, Point(0, 0).buffer(radius * rng.uniform(0.9, 1.3), 48)]).convex_hull
    hub = hull.difference(feet.buffer(wd * 0.5))
    hub = max(_polys(hub), key=lambda q: q.area) if _polys(hub) else Polygon()
    hub_wall = hub.exterior.buffer(wd / 2, cap_style=2, join_style=2).difference(feet.buffer(-wd * 0.1)) \
        if not hub.is_empty else Polygon()
    hub_room = hub.buffer(-wd / 2)
    plan = {"walls": unary_union([q["walls"] for q in placed] + [hub_wall]),
            "rooms": [r_ for q in placed for r_ in q["rooms"]] + ([("Entry/Hallway", q) for q in _polys(hub_room)]),
            "doors": [d for q in placed for d in q["doors"]], "windows": [w for q in placed for w in q["windows"]],
            "stairs": [s for q in placed for s in q["stairs"]],
            "inner": unary_union([q["inner"] for q in placed] + _polys(hub_room)), "mpu": mpu,
            "id": "syn_" + "_".join(p["id"] for p in parts) + "_{}".format(int(step))}
    # the hub's walls cut through nothing: openings where a wing meets the hub stay as the wing drew them
    return plan


# ----------------------------------------------------------------------------- drawing
def _style(rng):
    return {
        "wall": rng.choice(["solid", "solid", "hatch", "outline", "grey"]),
        "line": rng.choice([1, 1, 2, 2, 3]),
        "ink": int(rng.uniform(0, 70)),
        "paper": int(rng.uniform(236, 256)),
        "names": rng.random() < 0.85,
        "sizes": rng.random() < 0.6,
        "feet": rng.random() < 0.35,
        "dims": rng.random() < 0.5,
        "door_swing": rng.random() < 0.85,
        "window": rng.choice(["three", "two", "box"]),
        "blur": rng.random() < 0.3,
        "font": rng.choice([cv2.FONT_HERSHEY_SIMPLEX, cv2.FONT_HERSHEY_DUPLEX, cv2.FONT_HERSHEY_PLAIN,
                            cv2.FONT_HERSHEY_COMPLEX]),
    }


def _pts(poly, T):
    return np.round(T(np.asarray(poly.exterior.coords))).astype(np.int32)


def _fill(img, polys, T, val):
    for q in polys:
        if q.is_empty or q.area <= 0:
            continue
        cv2.fillPoly(img, [_pts(q, T)] + [np.round(T(np.asarray(h.coords))).astype(np.int32)
                                         for h in q.interiors], val)


def _feet(m):
    ft = m / 0.3048
    f, i = int(ft), int(round((ft - int(ft)) * 12))
    if i == 12:
        f, i = f + 1, 0
    return "{}'{}\"".format(f, i)


def render(plan, ppm, rng):
    """Draw `plan` at `ppm` pixels per metre. Returns (rgb image, room label, icon label)."""
    from shapely.ops import unary_union
    st = _style(rng)
    foot = unary_union([plan["walls"], plan["inner"]])
    minx, miny, maxx, maxy = foot.bounds
    k = plan["mpu"] * ppm                                    # pixels per unit
    margin = int(rng.uniform(0.8, 2.0) * ppm)
    W = int((maxx - minx) * k) + 2 * margin
    H = int((maxy - miny) * k) + 2 * margin
    if W * H > 9e6 or W < 64 or H < 64:
        return None
    T = lambda xy: np.stack([(xy[:, 0] - minx) * k + margin, (xy[:, 1] - miny) * k + margin], 1)
    paper, ink = st["paper"], st["ink"]
    img = np.full((H, W), paper, np.uint8)
    room = np.full((H, W), R["Background"], np.uint8)
    icon = np.full((H, W), I["NoIcon"], np.uint8)
    lw = max(1, int(round(st["line"] * ppm / 80)))

    # interior: untyped space is unknown, typed rooms get their class
    _fill(room, _polys(plan["inner"]), T, IGNORE)
    for cls, q in plan["rooms"]:
        _fill(room, _polys(q), T, R[cls])
    for q in plan["stairs"]:
        _fill(room, _polys(q), T, IGNORE)
        _fill(icon, _polys(q), T, I["Stairs"])

    # walls
    walls = _polys(plan["walls"])
    openings = [q for q in plan["doors"] + plan["windows"]]
    _fill(room, walls, T, R["Wall"])
    wall_px = np.zeros((H, W), np.uint8)
    _fill(wall_px, walls, T, 1)
    op_px = np.zeros((H, W), np.uint8)
    _fill(op_px, [q for o in openings for q in _polys(o)], T, 1)
    draw_wall = (wall_px > 0) & (op_px == 0)
    if st["wall"] == "solid":
        img[draw_wall] = ink
    elif st["wall"] == "grey":
        img[draw_wall] = int(rng.uniform(90, 170))
    elif st["wall"] == "hatch":
        hatch = np.zeros((H, W), np.uint8)
        sp = max(4, int(ppm * 0.08))
        for c in range(-H, W, sp):
            cv2.line(hatch, (c, 0), (c + H, H), 1, max(1, lw // 2))
        img[draw_wall & (hatch > 0)] = ink
    edges = cv2.morphologyEx(draw_wall.astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    img[edges] = ink
    if lw > 1:
        img[cv2.dilate(edges.astype(np.uint8), np.ones((lw, lw), np.uint8)) > 0] = ink

    # openings: the room head does not know whether an opening counts as wall -> IGNORE; icon head knows
    for q in [p for o in plan["windows"] for p in _polys(o)]:
        _fill(room, [q], T, IGNORE)
        _fill(icon, [q], T, I["Window"])
        rect = np.array(q.minimum_rotated_rectangle.exterior.coords)[:4]
        P = T(rect)
        e = [np.linalg.norm(P[(j + 1) % 4] - P[j]) for j in range(4)]
        j = int(np.argmax(e))
        a, b = P[j], P[(j + 1) % 4]
        n = P[(j + 2) % 4] - b
        cv2.polylines(img, [np.round(P).astype(np.int32)], True, ink, 1)
        lines = {"three": (0.25, 0.5, 0.75), "two": (0.35, 0.65), "box": ()}[st["window"]]
        for f in lines:
            cv2.line(img, tuple(np.round(a + n * f).astype(int)), tuple(np.round(b + n * f).astype(int)), ink, 1)
    for q in [p for o in plan["doors"] for p in _polys(o)]:
        _fill(room, [q], T, IGNORE)
        _fill(icon, [q], T, I["Door"])
        if not st["door_swing"]:
            continue
        rect = np.array(q.minimum_rotated_rectangle.exterior.coords)[:4]
        P = T(rect)
        e = [np.linalg.norm(P[(j + 1) % 4] - P[j]) for j in range(4)]
        j = int(np.argmax(e))
        a, b = P[j], P[(j + 1) % 4]
        width = np.linalg.norm(b - a)
        if width < 6:
            continue
        nrm = P[(j + 3) % 4] - a
        nrm = nrm / max(np.linalg.norm(nrm), 1e-6)
        side = 1 if rng.random() < 0.5 else -1                 # opens to one side or the other
        hinge, other = (a, b) if rng.random() < 0.5 else (b, a)
        leaf_dir = -side * nrm
        leaf = hinge + leaf_dir * width
        cv2.line(img, tuple(np.round(hinge).astype(int)), tuple(np.round(leaf).astype(int)), ink, max(1, lw))
        a0 = math.degrees(math.atan2(*(other - hinge)[::-1]))
        a1 = math.degrees(math.atan2(*(leaf - hinge)[::-1]))
        d = (a1 - a0 + 540) % 360 - 180
        cv2.ellipse(img, tuple(np.round(hinge).astype(int)), (int(width), int(width)), 0, a0, a0 + d, ink, 1)
        swing = np.zeros((H, W), np.uint8)
        cv2.ellipse(swing, tuple(np.round(hinge).astype(int)), (int(width), int(width)), 0, a0, a0 + d, 1, -1)
        icon[(swing > 0) & (icon == I["NoIcon"])] = IGNORE     # swing area: the icon head is not told

    # stairs: treads across the long side
    for q in [p for s in plan["stairs"] for p in _polys(s)]:
        rect = np.array(q.minimum_rotated_rectangle.exterior.coords)[:4]
        P = T(rect)
        e = [np.linalg.norm(P[(j + 1) % 4] - P[j]) for j in range(4)]
        j = int(np.argmax(e))
        a, b, c = P[j], P[(j + 1) % 4], P[(j + 2) % 4]
        nt = max(3, int(e[j] / max(ppm * 0.28, 3)))
        for t in range(nt + 1):
            u = a + (b - a) * t / nt
            cv2.line(img, tuple(np.round(u).astype(int)), tuple(np.round(u + (c - b)).astype(int)), ink, 1)
        cv2.polylines(img, [np.round(P).astype(np.int32)], True, ink, 1)

    # room names and sizes, dimension strings
    fs = max(0.3, ppm / 110) * rng.uniform(0.7, 1.2)
    for cls, q in plan["rooms"]:
        for p in _polys(q):
            if p.area * plan["mpu"] ** 2 < 2.0 or not st["names"]:
                continue
            c = T(np.array([[p.representative_point().x, p.representative_point().y]]))[0]
            name = rng.choice(ROOM_NAMES.get(cls, ["ROOM"]))
            (tw, th), _ = cv2.getTextSize(name, st["font"], fs, 1)
            org = (int(c[0] - tw / 2), int(c[1]))
            cv2.putText(img, name, org, st["font"], fs, ink, 1, cv2.LINE_AA)
            if st["sizes"]:
                bx = p.minimum_rotated_rectangle
                xy = np.array(bx.exterior.coords)[:3] * plan["mpu"]
                l1, l2 = np.linalg.norm(xy[1] - xy[0]), np.linalg.norm(xy[2] - xy[1])
                txt = "{} X {}".format(_feet(l1), _feet(l2)) if st["feet"] else "{:.2f} x {:.2f}".format(l1, l2)
                (tw2, _), _ = cv2.getTextSize(txt, st["font"], fs * 0.8, 1)
                cv2.putText(img, txt, (int(c[0] - tw2 / 2), int(c[1] + th * 1.8)), st["font"], fs * 0.8, ink, 1,
                            cv2.LINE_AA)
    if st["dims"]:
        x0, y0 = margin, int(margin * 0.45)
        x1 = W - margin
        cv2.line(img, (x0, y0), (x1, y0), ink, 1)
        for x in (x0, x1):
            cv2.line(img, (x - 6, y0 + 6), (x + 6, y0 - 6), ink, 1)
        L = (maxx - minx) * plan["mpu"]
        txt = _feet(L) if st["feet"] else "{:.2f} m".format(L)
        cv2.putText(img, txt, ((x0 + x1) // 2 - 30, y0 - 4), st["font"], fs, ink, 1, cv2.LINE_AA)

    if st["blur"]:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.4, 0.9))
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    return rgb, room, icon


# ----------------------------------------------------------------------------- cache writer
def write_entry(data_dir, sid, rgb, room, icon, max_side):
    h, w = room.shape
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        rgb = cv2.resize(rgb, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
        room = cv2.resize(room, (int(w * s), int(h * s)), interpolation=cv2.INTER_NEAREST)
        icon = cv2.resize(icon, (int(w * s), int(h * s)), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(os.path.join(data_dir, sid + "_img.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(data_dir, sid + "_lbl.png"), np.dstack([room, icon, np.zeros_like(room)]))


def build_rendered(cache_root, resplan_pkl, split_json=None, n_resplan=None, n_synthetic=3000, max_side=1536,
                   seed=0, log=print):
    """Render ResPlan plans and synthetic angled buildings into the cache. Returns index entries per split."""
    rng = random.Random(seed)
    data_dir = os.path.join(cache_root, "data")
    os.makedirs(data_dir, exist_ok=True)
    recs = load_resplan(resplan_pkl)
    split = {}
    if split_json and os.path.exists(split_json):
        sj = json.load(open(split_json))
        for name, ids in sj.items():
            for i in ids:
                split[int(i)] = "val" if name in ("val", "valid", "validation") else ("test" if name == "test"
                                                                                       else "train")
    out = {"train": [], "val": [], "test": []}
    recs = recs[:n_resplan] if n_resplan else recs
    plans = []
    for p in recs:
        try:
            pl = resplan_plan(p)
        except Exception:
            continue
        plans.append((split.get(int(p.get("id", -1)), "train" if rng.random() < 0.9 else "val"), pl))
    for n_done, (sp, pl) in enumerate(plans):
        res = render(pl, rng.uniform(55, 130), rng)
        if res is None:
            continue
        write_entry(data_dir, pl["id"], *res, max_side)
        out[sp].append({"id": pl["id"], "source": "resplan"})
        if n_done % 1000 == 0:
            log("  resplan rendered {}/{}".format(n_done, len(plans)))
    train_plans = [pl for sp, pl in plans if sp == "train"]
    val_plans = [pl for sp, pl in plans if sp == "val"]
    made = 0
    for t in range(n_synthetic * 3):
        if made >= n_synthetic:
            break
        sp = "val" if t % 10 == 0 else "train"
        pool = val_plans if sp == "val" and len(val_plans) > 3 else train_plans
        try:
            pl = synthetic_angled(rng.sample(pool, rng.choice([2, 2, 3])), rng)
            res = render(pl, rng.uniform(45, 100), rng)
        except Exception:
            continue
        if res is None:
            continue
        sid = "syn{:05d}".format(made)
        write_entry(data_dir, sid, *res, max_side)
        out[sp].append({"id": sid, "source": "synthetic"})
        made += 1
        if made % 500 == 0:
            log("  synthetic angled plans {}/{}".format(made, n_synthetic))
    return out


In [ ]:
%%writefile /kaggle/working/sheet_src/fp_extra_s3d.py
"""Structured3D (3,500 professionally designed apartments, 3D structure annotations) -> floor-plan model cache.

Each house's annotation_3d.json is turned into a drawn plan + label maps by training/s3d (s3d_convert: rooms,
walls with their real thickness, doors / windows; s3d_pairs.render: the drawing at 50 px/m in a random drafting
style). It teaches:
    room head   all room types, walls, background outside the flat
    icon head   Door, Window
    wall type   (derived in the data loader: walls the outside touches are exterior)
    level       every room floor is level 0 here: "Normal" (the counter-examples for raised / open-to-below)
No junction labels. Halls, warehouses and broken scenes are skipped (s3d_convert.is_junk).

Licence: Structured3D is for NON-COMMERCIAL research and education only and may NOT be redistributed in any
form. So this cache is rebuilt in every run from the copy attached to the notebook (a private Kaggle dataset)
and is never uploaded anywhere.
"""
import glob
import json
import os
import random
import sys
import zipfile
from multiprocessing import Pool

import cv2
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "s3d"))

LISTING = "s3d_index.json"


def find_root(search=("/kaggle/input", "/kaggle/working", "/tmp")):
    """The folder holding scene_xxxxx/annotation_3d.json, extracting the zip when only the zip is attached."""
    for base in search:
        for p in glob.glob(os.path.join(base, "**", "scene_00000", "annotation_3d.json"), recursive=True):
            return os.path.dirname(os.path.dirname(p))
    for base in search:
        for z in glob.glob(os.path.join(base, "**", "*tructured3D*annotation*3d*.zip"), recursive=True):
            out = os.path.join("/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp", "s3d_raw")
            if not os.path.exists(os.path.join(out, "Structured3D")):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(out)
            return os.path.join(out, "Structured3D")
    return None


def _one(job):
    root, scene, data_dir, max_side = job
    import s3d_convert
    import s3d_pairs
    try:
        res = s3d_convert.convert(os.path.join(root, scene, "annotation_3d.json"))
        if s3d_convert.is_junk(res):
            return None
        img, room, icon = s3d_pairs.render(res, hash(scene) & 0xffff)
        level = np.full(room.shape, 255, np.uint8)
        floor = (room != 0) & (room != 2)                     # every room (not background, not wall)
        level[floor] = 0
        h, w = room.shape
        s = min(1.0, max_side / max(h, w))
        if s < 1.0:
            size = (int(w * s), int(h * s))
            img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
            room, icon, level = [cv2.resize(m, size, interpolation=cv2.INTER_NEAREST) for m in (room, icon, level)]
        sid = "s3d_" + scene
        cv2.imwrite(os.path.join(data_dir, sid + "_img.png"), img)
        cv2.imwrite(os.path.join(data_dir, sid + "_lbl.png"), np.dstack([room, icon, level]))
        return sid
    except Exception:
        return None


def prepare_s3d(cache_root, max_side=1536, limit=None, val_fraction=0.05, seed=0, workers=None, log=print):
    """Add Structured3D to the cache; returns the merged index, or None when no Structured3D copy is attached."""
    from fp_prep import _merge_index
    listing = os.path.join(cache_root, LISTING)
    if os.path.exists(listing):
        return _merge_index(cache_root, json.load(open(listing)))
    root = find_root()
    if root is None:
        log("Structured3D not attached (add your private Kaggle dataset with Structured3D_annotation_3d.zip); "
            "training continues without it.")
        return None
    scenes = sorted(d for d in os.listdir(root) if d.startswith("scene_"))
    if limit:
        scenes = scenes[:limit]
    data_dir = os.path.join(cache_root, "data")
    os.makedirs(data_dir, exist_ok=True)
    log(f"Converting Structured3D: {len(scenes)} houses (drawn at 50 px/m; kept private, never uploaded) ...")
    jobs = [(root, s, data_dir, max_side) for s in scenes]
    ids = []
    with Pool(workers or os.cpu_count() or 2) as p:
        for k, sid in enumerate(p.imap_unordered(_one, jobs, chunksize=8), 1):
            if sid:
                ids.append(sid)
            if k % 500 == 0 or k == len(jobs):
                log(f"  structured3d {k}/{len(jobs)} ({len(ids)} kept)")
    ids.sort()
    rng = random.Random(seed)
    out = {"train": [], "val": []}
    for sid in ids:
        out["val" if rng.random() < val_fraction else "train"].append(
            {"id": sid, "source": "structured3d", "level": True})
    json.dump(out, open(listing, "w"))
    log(f"  structured3d: train {len(out['train'])}, val {len(out['val'])}")
    return _merge_index(cache_root, out)


In [ ]:
%%writefile /kaggle/working/sheet_src/s3d_convert.py
"""Structured3D annotation_3d.json -> 3D truth for PlanForge training (rooms, walls with thickness and type,
openings with sill / head / swing, storey heights). Units: metres in the output.

Walls:
  * solid walls = inside the outer wall faces and outside every room;
  * two rooms that touch with a wall plane on the shared edge (a zero-thickness wall in the source) get a
    partition of the thickness of the nearest door / window in that line, else 120 mm, cut from both rooms;
  * a shared edge with no wall plane is an open boundary (open plan) and stays open;
  * a room edge on the outer face (no outer wall drawn) gets an outer wall of the house's outer thickness,
    added outwards.
"""
import json
import numpy as np
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import unary_union, polygonize, linemerge

MM = 1000.0
DEFAULT_PART, DEFAULT_OUT = 120.0, 200.0
OPEN_TYPES = {"door", "window"}
SWING_PRIORITY = ["bathroom", "bedroom", "study", "kitchen", "store room", "laundry room", "office", "studio",
                  "balcony", "garden", "dining room", "corridor", "living room", "undefined"]


class Scene:
    def __init__(self, path):
        d = json.load(open(path))
        self.d = d
        self.J = np.array([j["coordinate"] for j in d["junctions"]], float)
        self.LJ = np.array(d["lineJunctionMatrix"], bool)
        self.PL = np.array(d["planeLineMatrix"], bool)
        self.planes = {p["ID"]: p for p in d["planes"]}

    def segments(self, pid, z=None):
        out = []
        for l in np.nonzero(self.PL[pid])[0]:
            js = np.nonzero(self.LJ[l])[0]
            if len(js) < 2:
                continue
            a, b = self.J[js[0]], self.J[js[1]]
            if z is not None and (abs(a[2] - z) > 1 or abs(b[2] - z) > 1):
                continue
            if np.linalg.norm(a[:2] - b[:2]) > 1:
                out.append(LineString([tuple(a[:2]), tuple(b[:2])]))
        return out

    def junctions(self, pid):
        return sorted({int(j) for l in np.nonzero(self.PL[pid])[0] for j in np.nonzero(self.LJ[l])[0]})


def _poly(segs):
    ps = list(polygonize(unary_union(segs))) if segs else []
    return unary_union(ps) if ps else None


def _on_plane(seg, plane, tol=6.0):
    n, off = np.array(plane["normal"][:2], float), float(plane["offset"])
    if abs(plane["normal"][2]) > 1e-6:
        return False
    return all(abs(float(np.dot(n, c)) + off) < tol for c in seg.coords)


def convert(path):
    S = Scene(path)
    rooms, openings, outwall = [], [], None
    for s in S.d["semantics"]:
        t = s["type"]
        if t in OPEN_TYPES:
            js = sorted({j for p in s["planeID"] for j in S.junctions(p)})
            P = S.J[js]
            xy = P[:, :2]
            c = xy.mean(0)
            _, _, vt = np.linalg.svd(xy - c)
            ax, nrm = vt[0], vt[1]
            pr, pn = (xy - c) @ ax, (xy - c) @ nrm
            mid = c + nrm * (pn.max() + pn.min()) / 2
            openings.append({"type": t, "a": mid + ax * pr.min(), "b": mid + ax * pr.max(),
                             "depth": float(pn.max() - pn.min()), "z0": float(P[:, 2].min()),
                             "z1": float(P[:, 2].max())})
        elif t == "outwall":
            outwall = s
        else:
            fl = [p for p in s["planeID"] if S.planes[p]["type"] == "floor"]
            ce = [p for p in s["planeID"] if S.planes[p]["type"] == "ceiling"]
            poly = _poly(S.segments(fl[0])) if fl else None
            if poly is None or poly.area < 1e4:                       # under 0.01 m2: noise
                continue
            rooms.append({"type": t, "poly": poly.buffer(0),
                          "walls": [S.planes[p] for p in s["planeID"] if S.planes[p]["type"] == "wall"],
                          "ceiling": abs(S.planes[ce[0]]["offset"]) if ce else 2800.0})
    if not rooms or outwall is None:
        raise ValueError("no rooms / no outer wall")
    footprint = _poly([g for p in outwall["planeID"] for g in S.segments(p, z=0.0)])
    if footprint is None:
        raise ValueError("no outer outline")
    allrooms = unary_union([r["poly"] for r in rooms])
    footprint = footprint.union(allrooms).buffer(0)

    def near_thickness(geom, default):
        best, bd = default, 400.0
        for o in openings:
            dd = geom.distance(LineString([tuple(o["a"]), tuple(o["b"])]))
            if dd < bd and 60 <= o["depth"] <= 400:
                best, bd = o["depth"], dd
        return best

    outer_t = [o["depth"] for o in openings if 60 <= o["depth"] <= 400 and
               footprint.boundary.distance(Point(*(o["a"] + o["b"]) / 2)) < 400]
    outer_t = float(np.median(outer_t)) if outer_t else DEFAULT_OUT
    added_part, added_out, open_edges = [], [], []
    # shared edges between rooms
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            A, B = rooms[i], rooms[j]
            sh = A["poly"].boundary.intersection(B["poly"].buffer(6))
            pieces = [g for g in getattr(sh, "geoms", [sh]) if g.geom_type in ("LineString", "MultiLineString")]
            for g in pieces:
                for seg in getattr(g, "geoms", [g]):
                    if seg.length < 50:
                        continue
                    walled = any(_on_plane(seg, p) for p in A["walls"] + B["walls"])
                    if walled:
                        t = near_thickness(seg, DEFAULT_PART)
                        added_part.append(seg.buffer(t / 2, cap_style=2, join_style=2))
                    else:
                        open_edges.append(seg)
    # room edges lying on the outer face: give them an outer wall, added outwards
    ext = footprint.boundary
    for r in rooms:
        on = r["poly"].boundary.intersection(ext.buffer(6))
        for seg in getattr(on, "geoms", [on]):
            if seg.geom_type == "LineString" and seg.length >= 50:
                added_out.append(seg.buffer(outer_t, cap_style=2, join_style=2).difference(allrooms))
    walls = footprint.difference(allrooms)
    if added_out:
        walls = walls.union(unary_union(added_out))
    if added_part:
        walls = walls.union(unary_union(added_part))
    walls = walls.buffer(0)
    outline = footprint.union(walls).buffer(1, join_style=2).buffer(-1, join_style=2)
    for r in rooms:
        r["poly"] = r["poly"].difference(walls).buffer(0)
    # openings: the rooms on each side, glass partitions, door swing
    for o in openings:
        a, b = np.asarray(o["a"]), np.asarray(o["b"])
        u = (b - a) / max(np.linalg.norm(b - a), 1e-9)
        n = np.array([-u[1], u[0]])
        m = (a + b) / 2
        d = o["depth"] / 2 + 250
        side = {}
        for k in (1, -1):
            pt = Point(*(m + n * d * k))
            side[k] = next((r["type"] for r in rooms if r["poly"].contains(pt)), None)
        o["rooms"] = [side[1], side[-1]]
        if o["type"] == "window" and side[1] and side[-1] and o["z0"] < 50:
            o["type"] = "glass_partition"
        elif o["type"] == "window" and side[1] and side[-1]:
            o["type"] = "internal_window"
        if o["type"] == "door":
            cands = [k for k in (1, -1) if side[k]]
            if len(cands) == 2:
                cands.sort(key=lambda k: SWING_PRIORITY.index(side[k]) if side[k] in SWING_PRIORITY else 99)
            k = cands[0] if cands else 1                              # outside -> swings in (no room: k=1)
            if not side[k] and side[-k]:
                k = -k
            w = np.linalg.norm(b - a)
            best = None
            for hinge, other in ((a, b), (b, a)):
                arc = Point(*hinge).buffer(w).intersection(
                    Polygon([hinge, other, other + n * k * w, hinge + n * k * w]))
                clash = arc.intersection(walls).area
                if best is None or clash < best[0]:
                    best = (clash, hinge, other)
            o["swing"] = int(k)
            o["hinge"] = best[1]
    return {"rooms": rooms, "openings": openings, "walls": walls, "outline": outline,
            "open_edges": open_edges, "outer_t": outer_t,
            "ceiling": float(np.median([r["ceiling"] for r in rooms]))}


def is_junk(res):
    """Halls, warehouses and broken scenes that are not homes."""
    area = res["outline"].area / 1e6
    big = max(r["poly"].area for r in res["rooms"]) / 1e6
    named = sum(r["poly"].area for r in res["rooms"] if r["type"] != "undefined") / max(
        sum(r["poly"].area for r in res["rooms"]), 1)
    return area > 400 or big > 150 or (len(res["rooms"]) < 2) or (named < 0.3 and area > 80)


def to_truth(res, scene_id):
    """3D truth in metres: what the model must reproduce."""
    def P(g):
        return [[[round(x / MM, 4), round(y / MM, 4)] for x, y in p.exterior.coords]
                for p in getattr(g, "geoms", [g]) if p.area > 100]
    return {"scene": scene_id, "units": "m", "storey": {"floor_z": 0.0, "ceiling_z": round(res["ceiling"] / MM, 3)},
            "outer_wall_thickness": round(res["outer_t"] / MM, 3),
            "rooms": [{"type": r["type"], "polygon": P(r["poly"]), "area": round(r["poly"].area / 1e6, 3),
                       "ceiling_z": round(r["ceiling"] / MM, 3)} for r in res["rooms"]],
            "walls": P(res["walls"]),
            "openings": [{"type": o["type"], "a": [round(v / MM, 4) for v in o["a"]],
                          "b": [round(v / MM, 4) for v in o["b"]], "thickness": round(o["depth"] / MM, 3),
                          "sill": round(o["z0"] / MM, 3), "head": round(o["z1"] / MM, 3),
                          "rooms": o["rooms"], **({"swing": o["swing"], "hinge": [round(v / MM, 4) for v in o["hinge"]]}
                                                  if "swing" in o else {})} for o in res["openings"]],
            "open_edges": [[[round(x / MM, 4), round(y / MM, 4)] for x, y in e.coords] for e in res["open_edges"]]}


In [ ]:
%%writefile /kaggle/working/sheet_src/s3d_pairs.py
"""Batch: every Structured3D house -> (plan drawing PNG, 3D truth JSON, label masks NPZ).
Drawing at a fixed 50 px per metre with randomised drafting style; masks use PlanForge's room / icon classes."""
import os, sys, json, random, traceback
import numpy as np
import cv2
from multiprocessing import Pool
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import s3d_convert as conv2

ROOT = os.environ.get("S3D_ROOT", "/kaggle/input/structured3d/Structured3D")
OUT = os.environ.get("S3D_OUT", "/kaggle/working/s3d_pairs")
PPM = 50.0                                     # pixels per metre
PAD = 1.0                                      # m of paper around the building
ROOMS = ["Background", "Outdoor", "Wall", "Kitchen", "LivingRoom", "Bedroom", "Bath", "Entry/Hallway",
         "Railing", "Storage", "Garage", "OtherRoom"]
ICON_WIN, ICON_DOOR = 1, 2
TYPE_TO_ROOM = {"kitchen": "Kitchen", "living room": "LivingRoom", "dining room": "LivingRoom",
                "bedroom": "Bedroom", "study": "Bedroom", "studio": "LivingRoom", "office": "OtherRoom",
                "bathroom": "Bath", "laundry room": "Storage", "store room": "Storage", "corridor": "Entry/Hallway",
                "balcony": "Outdoor", "garden": "Outdoor", "undefined": "OtherRoom"}


def render(res, seed):
    rng = random.Random(seed)
    x0, y0, x1, y1 = res["outline"].bounds
    W = int((x1 - x0) / 1000 * PPM + 2 * PAD * PPM)
    H = int((y1 - y0) / 1000 * PPM + 2 * PAD * PPM)
    if W * H > 4000 * 4000:
        raise ValueError("too large")

    def px(pts):                                  # mm -> px (y down)
        p = np.asarray(pts, float)
        return np.stack([(p[:, 0] - x0) / 1000 * PPM + PAD * PPM, (y1 - p[:, 1]) / 1000 * PPM + PAD * PPM], 1)

    paper = rng.choice([255, 250, 246])
    img = np.full((H, W, 3), paper, np.uint8)
    room = np.zeros((H, W), np.uint8)
    icon = np.zeros((H, W), np.uint8)
    lw = rng.choice([1, 1, 2])
    ink = (rng.randint(0, 40),) * 3
    style = rng.choice(["solid", "solid", "grey", "hatch", "outline"])

    def polys(g):
        return [p for p in getattr(g, "geoms", [g]) if p.area > 100]

    # rooms (masks)
    for r in res["rooms"]:
        cls = ROOMS.index(TYPE_TO_ROOM.get(r["type"], "OtherRoom"))
        for p in polys(r["poly"]):
            cv2.fillPoly(room, [np.round(px(p.exterior.coords)).astype(np.int32)], cls)
    # walls
    wall_px = np.zeros((H, W), np.uint8)
    for p in polys(res["walls"]):
        ext = np.round(px(p.exterior.coords)).astype(np.int32)
        holes = [np.round(px(h.coords)).astype(np.int32) for h in p.interiors]
        cv2.fillPoly(wall_px, [ext] + holes, 1)
    room[wall_px > 0] = ROOMS.index("Wall")
    if style in ("solid", "grey"):
        img[wall_px > 0] = ink if style == "solid" else (120, 120, 120)
    elif style == "hatch":
        yy, xx = np.mgrid[0:H, 0:W]
        hatch = ((xx + yy) % 8 < 2)
        img[(wall_px > 0) & hatch] = ink
    edges = cv2.morphologyEx(wall_px, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))
    if style != "solid":
        img[edges > 0] = ink
    # openings
    for o in res["openings"]:
        a, b = np.asarray(o["a"]), np.asarray(o["b"])
        w = np.linalg.norm(b - a)
        if w < 1:
            continue
        u = (b - a) / w
        n = np.array([-u[1], u[0]])
        h = o["depth"] / 2 + 20
        quad = np.round(px([a - n * h, b - n * h, b + n * h, a + n * h])).astype(np.int32)
        cv2.fillPoly(img, [quad], (paper,) * 3)
        cls = ICON_DOOR if o["type"] == "door" else ICON_WIN
        cv2.fillPoly(icon, [quad], cls)
        room[cv2.fillPoly(np.zeros_like(room), [quad], 1) > 0] = ROOMS.index("Wall")
        if o["type"] in ("window", "internal_window"):
            for k in (-0.6, 0, 0.6):
                q = np.round(px([a + n * h * k, b + n * h * k])).astype(np.int32)
                cv2.line(img, tuple(q[0]), tuple(q[1]), ink, lw)
            for e in (a, b):
                q = np.round(px([e - n * h, e + n * h])).astype(np.int32)
                cv2.line(img, tuple(q[0]), tuple(q[1]), ink, lw)
        elif o["type"] == "glass_partition":
            for k in (-0.25, 0.25):
                q = np.round(px([a + n * h * k, b + n * h * k])).astype(np.int32)
                cv2.line(img, tuple(q[0]), tuple(q[1]), ink, 1)
        else:
            k = o["swing"]
            hinge = np.asarray(o["hinge"])
            other = b if np.allclose(hinge, a) else a
            leaf_end = hinge + n * k * w
            q = np.round(px([hinge, leaf_end])).astype(np.int32)
            cv2.line(img, tuple(q[0]), tuple(q[1]), ink, lw)
            t = np.linspace(0, np.pi / 2, 24)
            d0 = (other - hinge) / w
            arc = [hinge + w * (np.cos(tt) * d0 + np.sin(tt) * n * k) for tt in t]
            cv2.polylines(img, [np.round(px(arc)).astype(np.int32)], False, ink, 1)
    # room names (as many plans have)
    if rng.random() < 0.8:
        for r in res["rooms"]:
            p = max(polys(r["poly"]), key=lambda q: q.area, default=None)
            if p is None or p.area < 3e6:
                continue
            c = px([p.representative_point().coords[0]])[0]
            txt = r["type"].upper() if r["type"] != "undefined" else ""
            sc = 0.35 if p.area > 8e6 else 0.28
            (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, sc, 1)
            cv2.putText(img, txt, (int(c[0] - tw / 2), int(c[1] + th / 2)), cv2.FONT_HERSHEY_SIMPLEX, sc, ink, 1,
                        cv2.LINE_AA)
    return img, room, icon


def one(sc):
    try:
        res = conv2.convert(os.path.join(ROOT, sc, "annotation_3d.json"))
        if conv2.is_junk(res):
            return sc, "junk"
        img, room, icon = render(res, hash(sc) & 0xffff)
        cv2.imwrite(os.path.join(OUT, "img", sc + ".png"), img)
        np.savez_compressed(os.path.join(OUT, "mask", sc + ".npz"), room=room, icon=icon)
        truth = conv2.to_truth(res, sc)
        truth["drawing"] = {"px_per_m": PPM, "pad_m": PAD, "origin_mm": [res["outline"].bounds[0], res["outline"].bounds[3]],
                            "size_px": [img.shape[1], img.shape[0]]}
        json.dump(truth, open(os.path.join(OUT, "truth", sc + ".json"), "w"))
        return sc, "ok"
    except Exception as e:
        return sc, "error: " + repr(e)[:100]


if __name__ == "__main__":
    for d in ("img", "mask", "truth"):
        os.makedirs(os.path.join(OUT, d), exist_ok=True)
    scenes = sorted(os.listdir(ROOT))
    with Pool(os.cpu_count()) as p:
        res = p.map(one, scenes, chunksize=10)
    import collections
    c = collections.Counter(s.split(":")[0] for _, s in res)
    print(c)
    print(collections.Counter(s for _, s in res if s.startswith("error")).most_common(5))
    json.dump(dict(res), open(os.path.join(OUT, "status.json"), "w"))


In [ ]:
%%writefile /kaggle/working/sheet_src/compose_sheets.py
"""Multi-view drawing sheets from real plans, for the sheet-view detector (head 0).

Architects often put the ground floor, first floor and terrace plans of one building side by side on one sheet,
with a title under each view, a border, a title block, notes and schedules. This composes such sheets from real
plan drawings (ResPlan renders, Structured3D renders, CubiCasa5K scans: anything drawn), so the position of
every view is known exactly:

    plan_view       the drawing of one floor (not its title)
    title_block     the drawing's title block
    table           door / window schedules and area tables

Titles are taken from the names real sheets use (GROUND FLOOR PLAN, FIRST FLOOR PLAN, TERRACE PLAN ...). The
plans on one composed sheet are different buildings: these sheets teach FINDING the views, not stacking floors.
"""
import math
import os
import random

import cv2
import numpy as np

CLASSES = ["plan_view", "title_block", "table"]
TITLES = ["GROUND FLOOR PLAN", "FIRST FLOOR PLAN", "SECOND FLOOR PLAN", "TERRACE PLAN", "STILT FLOOR PLAN",
          "BASEMENT PLAN", "TYPICAL FLOOR PLAN", "GF PLAN", "FF PLAN", "SF PLAN", "ROOF PLAN", "MEZZANINE PLAN"]
WORDS = ["PROPOSED", "RESIDENCE", "BUILDING", "FOR", "MR", "SITE", "NO", "PLOT", "SURVEY", "DRAWN", "CHECKED",
         "APPROVED", "ARCHITECT", "SCALE", "DATE", "SHEET", "REV", "NOTES", "ALL", "DIMENSIONS", "ARE", "IN", "MM",
         "WALLS", "230", "THK", "BRICK", "LINTEL", "LEVEL", "PLINTH", "+0.45", "FFL", "1:100", "NTS", "CLIENT"]
FONTS = [cv2.FONT_HERSHEY_SIMPLEX, cv2.FONT_HERSHEY_DUPLEX, cv2.FONT_HERSHEY_COMPLEX, cv2.FONT_HERSHEY_TRIPLEX,
         cv2.FONT_HERSHEY_PLAIN]


def content_box(rgb, room=None, pad=4):
    """Bounding box of the drawing: labelled building pixels when given, else ink."""
    if room is not None:
        m = (room != 0) & (room != 255) & (room != 254)
    else:
        m = rgb.min(axis=2) < 200
    ys, xs = np.nonzero(m)
    if len(xs) < 20:
        return None
    x0, y0 = max(0, xs.min() - pad), max(0, ys.min() - pad)
    x1, y1 = min(rgb.shape[1], xs.max() + pad + 1), min(rgb.shape[0], ys.max() + pad + 1)
    return x0, y0, x1, y1


def _text(img, s, x, y, scale, rng, ink):
    f = rng.choice(FONTS)
    th = max(1, int(round(scale * rng.uniform(1.2, 2.2))))
    cv2.putText(img, s, (int(x), int(y)), f, scale, ink, th, cv2.LINE_AA)
    (w, h), _ = cv2.getTextSize(s, f, scale, th)
    return w, h


def _sentence(rng, n):
    return " ".join(rng.choice(WORDS) for _ in range(n))


def _table(img, x, y, w, h, rng, ink):
    rows, cols = rng.randint(3, 9), rng.randint(2, 5)
    cv2.rectangle(img, (x, y), (x + w, y + h), ink, 2)
    for r in range(1, rows):
        yy = y + r * h // rows
        cv2.line(img, (x, yy), (x + w, yy), ink, 1)
    for c in range(1, cols):
        xx = x + c * w // cols
        cv2.line(img, (xx, y), (xx, y + h), ink, 1)
    sc = max(0.3, min(0.9, h / rows / 40))
    for r in range(rows):
        for c in range(cols):
            if rng.random() < 0.8:
                cell = rng.choice(["D1", "D2", "W1", "W2", "V1", "1000", "900", "2100", "1200", "MAIN DOOR",
                                   "WINDOW", "VENT", str(rng.randint(1, 30))])
                cv2.putText(img, cell, (x + c * w // cols + 4, y + r * h // rows + int(h / rows * 0.7)),
                            cv2.FONT_HERSHEY_SIMPLEX, sc, ink, 1, cv2.LINE_AA)


def _north(img, cx, cy, r, ink):
    cv2.circle(img, (cx, cy), r, ink, 2)
    pts = np.array([[cx, cy - r], [cx - r // 3, cy + r // 2], [cx, cy + r // 5], [cx + r // 3, cy + r // 2]], np.int32)
    cv2.fillPoly(img, [pts], ink)
    cv2.putText(img, "N", (cx - r // 4, cy - r - 6), cv2.FONT_HERSHEY_SIMPLEX, r / 30, ink, 2)


def compose(plans, rng, size=None):
    """plans: list of (rgb uint8, room labels or None). Returns (sheet rgb, [(class, x0, y0, x1, y1)])."""
    W = size or rng.choice([2400, 2800, 3300, 4000])
    H = int(W / rng.choice([1.414, 1.414, 1.5, 1.3]))
    paper = rng.choice([255, 252, 248])
    ink = (rng.randint(0, 40),) * 3
    img = np.full((H, W, 3), paper, np.uint8)
    boxes = []
    m = int(W * rng.uniform(0.015, 0.03))
    if rng.random() < 0.85:                                     # border frame
        cv2.rectangle(img, (m, m), (W - m, H - m), ink, rng.choice([2, 3, 4]))
        if rng.random() < 0.5:
            cv2.rectangle(img, (m + 10, m + 10), (W - m - 10, H - m - 10), ink, 1)
    # title block: bottom strip or right column
    right = rng.random() < 0.55
    if right:
        tbw = int(W * rng.uniform(0.14, 0.2))
        tb = (W - m - tbw, m, W - m, H - m)
        area = (m + 20, m + 20, W - m - tbw - 20, H - m - 20)
    else:
        tbh = int(H * rng.uniform(0.1, 0.15))
        tb = (m, H - m - tbh, W - m, H - m)
        area = (m + 20, m + 20, W - m - 20, H - m - tbh - 20)
    if rng.random() < 0.9:
        x0, y0, x1, y1 = tb
        cv2.rectangle(img, (x0, y0), (x1, y1), ink, 3)
        n = rng.randint(3, 7)
        for k in range(1, n):
            if right:
                yy = y0 + int((y1 - y0) * (0.5 + 0.5 * k / n))
                cv2.line(img, (x0, yy), (x1, yy), ink, 1)
            else:
                xx = x0 + int((x1 - x0) * (0.4 + 0.6 * k / n))
                cv2.line(img, (xx, y0), (xx, y1), ink, 1)
        for k in range(rng.randint(4, 10)):
            sc = rng.uniform(0.6, 1.2) * W / 3300
            tx = x0 + 12 + (rng.randint(0, max(1, (x1 - x0) // 3)) if not right else 0)
            ty = y0 + 30 + k * int(40 * W / 3300)
            if ty < y1 - 8:
                _text(img, _sentence(rng, rng.randint(1, 4)), tx, ty, sc, rng, ink)
        boxes.append(("title_block", x0, y0, x1, y1))
    # notes / tables in a side band of the drawing area, sometimes
    ax0, ay0, ax1, ay1 = area
    if rng.random() < 0.5:
        bw = int((ax1 - ax0) * rng.uniform(0.16, 0.24))
        bx = ax1 - bw
        y = ay0 + 10
        if rng.random() < 0.7:
            th = int((ay1 - ay0) * rng.uniform(0.15, 0.3))
            _table(img, bx, y, bw, th, rng, ink)
            boxes.append(("table", bx, y, bx + bw, y + th))
            y += th + 30
        for k in range(rng.randint(3, 10)):
            sc = 0.7 * W / 3300
            if y + 30 > ay1:
                break
            _text(img, "{}. {}".format(k + 1, _sentence(rng, rng.randint(3, 6))), bx, y + 20, sc, rng, ink)
            y += int(34 * W / 3300)
        ax1 = bx - 30
    # views in a grid
    n = len(plans)
    cols = n if n <= 3 and (ax1 - ax0) > 1.3 * (ay1 - ay0) else math.ceil(math.sqrt(n))
    rows = math.ceil(n / cols)
    cw, ch = (ax1 - ax0) // cols, (ay1 - ay0) // rows
    titles = rng.sample(TITLES[:4], min(n, 4)) if n <= 4 and rng.random() < 0.7 else \
        [rng.choice(TITLES) for _ in range(n)]
    for k, (rgb, room) in enumerate(plans):
        r, c = divmod(k, cols)
        cb = content_box(rgb, room)
        if cb is None:
            continue
        x0, y0, x1, y1 = cb
        crop = rgb[y0:y1, x0:x1]
        tsp = int(ch * 0.12)                                       # room for the title under the view
        s = min((cw * 0.92) / crop.shape[1], (ch * 0.92 - tsp) / crop.shape[0]) * rng.uniform(0.7, 1.0)
        if s <= 0:
            continue
        crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
        hh, ww = crop.shape[:2]
        px = ax0 + c * cw + (cw - ww) // 2 + rng.randint(-cw // 20, cw // 20)
        py = ay0 + r * ch + (ch - tsp - hh) // 2 + rng.randint(-ch // 30, ch // 30)
        px, py = max(ax0, min(px, ax0 + (c + 1) * cw - ww)), max(ay0, py)
        if py + hh > H or px + ww > W:
            continue
        region = img[py:py + hh, px:px + ww]
        np.minimum(region, crop, out=region)                       # ink over paper
        boxes.append(("plan_view", px, py, px + ww, py + hh))
        if rng.random() < 0.9:                                      # its title
            sc = rng.uniform(0.9, 1.6) * W / 3300
            tw, th_ = cv2.getTextSize(titles[k], cv2.FONT_HERSHEY_DUPLEX, sc, 2)[0]
            tx, ty = px + ww // 2 - tw // 2, py + hh + int(tsp * 0.6)
            if ty < H - m:
                _text(img, titles[k], tx, ty, sc, rng, ink)
                if rng.random() < 0.6:
                    cv2.line(img, (tx, ty + 8), (tx + tw, ty + 8), ink, 2)
                if rng.random() < 0.5:
                    _text(img, "SCALE 1:{}".format(rng.choice([50, 100, 150, 200])), tx, ty + int(40 * W / 3300),
                          sc * 0.6, rng, ink)
    if rng.random() < 0.6:
        _north(img, ax0 + 60, ay0 + 70, int(40 * W / 3300), ink)
    return img, boxes


def yolo_lines(boxes, W, H):
    out = []
    for cls, x0, y0, x1, y1 in boxes:
        cx, cy, w, h = (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H
        if w > 0.005 and h > 0.005:
            out.append("{} {:.6f} {:.6f} {:.6f} {:.6f}".format(CLASSES.index(cls), cx, cy, w, h))
    return out


In [ ]:
%%writefile /kaggle/working/sheet_src/train_sheets.py
import json, sys
from ultralytics import YOLO
cfg = json.load(open(sys.argv[1]))
YOLO(cfg["model"]).train(
    data=cfg["data"], imgsz=cfg["imgsz"], epochs=cfg["epochs"], time=cfg["hours"], batch=cfg["batch"],
    device=cfg["device"], workers=4, project=cfg["project"], name="det", exist_ok=True, patience=40, cos_lr=True,
    seed=42, amp=True, plots=True,
    # drawing sheets are upright and carry text: no rotation / mirror; mild scale and translate only
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, degrees=0, fliplr=0.0, flipud=0.0, mosaic=0.3, scale=0.25, translate=0.05,
    close_mosaic=10)


In [ ]:
# ===================== SANITY + HF =====================
def fail(msg): raise RuntimeError("\n\n*** SETUP PROBLEM: " + msg + " ***\n")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus)) if gpus else fail("No GPU: Settings -> Accelerator -> GPU T4 x2.")
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Kaggle secret not readable:", e)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if not HF_TOKEN: fail("Add a Kaggle Secret named HF_TOKEN and attach it to this notebook.")
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
MODEL_REPO = f"{api.whoami()['name']}/{MODEL_REPO_NAME}"
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
print("HF repo:", MODEL_REPO)

In [ ]:
# ===================== PLAN DRAWINGS (the views) =====================
sys.path.insert(0, SRC_DIR)
import cv2, numpy as np
import fp_render
rng = random.Random(42)
t0 = time.time()
if not os.path.exists("/tmp/ResPlan/ResPlan.pkl"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/m-agour/ResPlan", "/tmp/ResPlan"], check=True)
    import zipfile; zipfile.ZipFile("/tmp/ResPlan/ResPlan.zip").extractall("/tmp/ResPlan")
recs = fp_render.load_resplan("/tmp/ResPlan/ResPlan.pkl")
rng.shuffle(recs)
plans = []                                       # (png path, has room labels)
for k, p in enumerate(recs[:N_RESPLAN]):
    try:
        res = fp_render.render(fp_render.resplan_plan(p), rng.uniform(40, 110), rng)
    except Exception:
        continue
    if res is None: continue
    rgb, room, _ = res
    fn = os.path.join(PLAN_DIR, f"rp{k:05d}")
    cv2.imwrite(fn + "_img.png", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)); cv2.imwrite(fn + "_room.png", room)
    plans.append(fn)
    if len(plans) % 1000 == 0: print(f"  resplan views {len(plans)}")
import fp_extra_s3d
s3d_root = fp_extra_s3d.find_root()
if s3d_root:
    import s3d_convert, s3d_pairs
    scenes = sorted(d for d in os.listdir(s3d_root) if d.startswith("scene_"))[:N_S3D]
    for sc in scenes:
        try:
            res = s3d_convert.convert(os.path.join(s3d_root, sc, "annotation_3d.json"))
            if s3d_convert.is_junk(res): continue
            img, room, _ = s3d_pairs.render(res, rng.randrange(1 << 16))
        except Exception:
            continue
        fn = os.path.join(PLAN_DIR, "s3d_" + sc)
        cv2.imwrite(fn + "_img.png", img); cv2.imwrite(fn + "_room.png", room)
        plans.append(fn)
    print("  + Structured3D views (private, not uploaded)")
else:
    print("  Structured3D not attached: ResPlan views only")
print(f"{len(plans)} plan drawings in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ===================== COMPOSE SHEETS =====================
import compose_sheets as C
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    os.makedirs(os.path.join(DS_DIR, sub), exist_ok=True)
def load(fn):
    img = cv2.cvtColor(cv2.imread(fn + "_img.png"), cv2.COLOR_BGR2RGB)
    return img, cv2.imread(fn + "_room.png", cv2.IMREAD_UNCHANGED)
counts = {c: 0 for c in C.CLASSES}
t0 = time.time()
for k in range(N_SHEETS):
    split = "val" if k % 10 == 0 else "train"
    pool = plans[: max(1, len(plans) // 10)] if split == "val" else plans[len(plans) // 10:]   # no plan in both
    n = min(rng.choice([1, 2, 2, 2, 3, 3, 4]), len(pool))
    sheet, boxes = C.compose([load(f) for f in rng.sample(pool, n)], rng)
    H, W = sheet.shape[:2]
    s = SHEET_MAX_SIDE / max(H, W)
    sheet = cv2.resize(sheet, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    if rng.random() < 0.3:                                   # scanned look
        sheet = np.clip(sheet.astype(np.float32) + np.random.normal(0, rng.uniform(2, 6), sheet.shape), 0, 255).astype(np.uint8)
    stem = f"sheet{k:05d}"
    cv2.imwrite(os.path.join(DS_DIR, f"images/{split}/{stem}.jpg"), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, rng.randint(70, 95)])
    open(os.path.join(DS_DIR, f"labels/{split}/{stem}.txt"), "w").write("\n".join(C.yolo_lines(boxes, W, H)) + "\n")
    for b in boxes: counts[b[0]] += 1
    if (k + 1) % 1000 == 0: print(f"  sheets {k + 1}/{N_SHEETS}")
open(os.path.join(DS_DIR, "data.yaml"), "w").write(
    f"path: {DS_DIR}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(C.CLASSES)))
shutil.rmtree(PLAN_DIR, ignore_errors=True)                  # Structured3D renders are deleted here
print("boxes:", counts, f"| {(time.time()-t0)/60:.1f} min")

In [ ]:
# ===================== TRAIN =====================
hours = (NOTEBOOK_START + TIME_BUDGET_HOURS * 3600 - time.time()) / 3600 - 0.3
ngpu = len(gpus)
cfg = {"model": BASE_MODEL, "data": os.path.join(DS_DIR, "data.yaml"), "imgsz": IMGSZ, "epochs": MAX_EPOCHS,
       "hours": round(hours, 2), "batch": BATCH if ngpu > 1 else BATCH // 2, "device": list(range(ngpu)) if ngpu > 1 else 0,
       "project": RUNS_DIR}
json.dump(cfg, open(os.path.join(SRC_DIR, "run.json"), "w"))
p = subprocess.Popen([sys.executable, os.path.join(SRC_DIR, "train_sheets.py"), os.path.join(SRC_DIR, "run.json")],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd="/tmp")
for line in p.stdout:
    line = line.replace("\r", "\n").rstrip().split("\n")[-1]
    if line and not ("it/s" in line and "100%" not in line): print(line, flush=True)
print("trainer exit code", p.wait())

In [ ]:
# ===================== EXPORT + UPLOAD =====================
from ultralytics import YOLO
W_DIR = os.path.join(RUNS_DIR, "det", "weights")
best = os.path.join(W_DIR, "best.pt")
m = YOLO(best)
r = m.val(data=os.path.join(DS_DIR, "data.yaml"), imgsz=IMGSZ, batch=8, device=0, plots=False)
onnx_path = m.export(format="onnx", imgsz=IMGSZ, dynamic=False, simplify=True, half=False)
meta = {"classes": C.CLASSES, "imgsz": IMGSZ, "sheet_max_side": SHEET_MAX_SIDE,
        "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
        "per_class_mAP50_95": {c: float(v) for c, v in zip(C.CLASSES, r.box.maps)},
        "data": "composed sheets from ResPlan" + (" + Structured3D (non-commercial)" if s3d_root else "")}
json.dump(meta, open("/tmp/sheet_views_meta.json", "w"), indent=1)
for src, dst in ((best, "checkpoints/best.pt"), (onnx_path, "sheet_views.onnx"), ("/tmp/sheet_views_meta.json", "sheet_views_meta.json")):
    api.upload_file(path_or_fileobj=src, path_in_repo=dst, repo_id=MODEL_REPO, commit_message="sheet-view detector")
print(json.dumps(meta, indent=1))
print(f"model: https://huggingface.co/{MODEL_REPO}")